In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 11_predict_premium_customers
# MAGIC Cargar datos de inferencia, aplicar modelo y generar predicciones

# COMMAND ----------

import pandas as pd
import mlflow
import mlflow.sklearn
from pyspark.sql import functions as F
from datetime import datetime

INFERENCE_PATH = "/Volumes/olist/olist_gold/inference/"
RESULTS_PATH = "/Volumes/olist/olist_gold/predictions/"

# ⚠️ CONFIGURAR: Run ID del mejor modelo (desde notebook 08)
BEST_MODEL_RUN_ID = "aa1ead44843d4b70bc9abbf9e9ec479c"  # ← REEMPLAZAR

start_time = datetime.now()
print("🚀 Iniciando predicción de clientes premium\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Listar Datasets de Inferencia Disponibles

# COMMAND ----------

print("📂 ETAPA 1: DATASETS DISPONIBLES\n" + "="*60 + "\n")

try:
    inference_datasets = dbutils.fs.ls(INFERENCE_PATH)
    
    print("📊 Datasets de inferencia encontrados:\n")
    for i, dataset in enumerate(inference_datasets, 1):
        if dataset.isDir() and 'customer_features_pca_' in dataset.name:
            size_mb = dataset.size / (1024 * 1024) if dataset.size else 0
            print(f"{i}. {dataset.name}")
    
    print()
except Exception as e:
    print(f"❌ Error: {e}")
    raise FileNotFoundError("No se encontraron datasets de inferencia")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Seleccionar Dataset (Automático: más reciente)

# COMMAND ----------

print("🔍 ETAPA 2: SELECCIÓN DE DATASET\n" + "="*60 + "\n")

# Filtrar solo directorios con customer_features_pca
pca_datasets = [d for d in inference_datasets if d.isDir() and 'customer_features_pca_' in d.name]

if not pca_datasets:
    raise FileNotFoundError("No hay datasets PCA disponibles")

# Ordenar por nombre (contiene fecha) y tomar el último
latest_dataset = sorted(pca_datasets, key=lambda x: x.name)[0]
dataset_name = latest_dataset.name.rstrip('/')
dataset_path = f"{INFERENCE_PATH}{dataset_name}/"

print(f"✅ Dataset seleccionado: {dataset_name}")
print(f"📂 Ruta: {dataset_path}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Cargar Dataset de Inferencia

# COMMAND ----------

print("📥 ETAPA 3: CARGA DE DATOS\n" + "="*60 + "\n")

# Cargar dataset PCA
customer_pca_df = spark.read.format("delta").load(dataset_path).toPandas()

print(f"✅ Dataset cargado: {customer_pca_df.shape}")
print(f"   • Clientes: {len(customer_pca_df):,}")
print(f"   • Columnas: {len(customer_pca_df.columns)}")
print(f"\n📋 Columnas: {list(customer_pca_df.columns)}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Cargar Modelo Entrenado

# COMMAND ----------

print("🤖 ETAPA 4: CARGA DE MODELO\n" + "="*60 + "\n")

print(f"📦 Cargando modelo desde MLflow...")
print(f"   Run ID: {BEST_MODEL_RUN_ID}\n")

try:
    model = mlflow.sklearn.load_model(f"runs:/{BEST_MODEL_RUN_ID}/model")
    print("✅ Modelo cargado exitosamente\n")
except Exception as e:
    print(f"❌ Error al cargar modelo: {e}")
    print("\n💡 Verifica:")
    print("   1. El run_id es correcto")
    print("   2. El modelo existe en MLflow")
    print("   3. Tienes permisos de acceso")
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Preparar Datos para Predicción

# COMMAND ----------

print("🔧 ETAPA 5: PREPARACIÓN DE DATOS\n" + "="*60 + "\n")

# Separar customer_id de features
customer_ids = customer_pca_df['customer_id'].copy()
pca_cols = [c for c in customer_pca_df.columns if c.startswith('pca_')]

X_inference = customer_pca_df[pca_cols].copy()

print(f"✅ Datos preparados:")
print(f"   • Features PCA: {len(pca_cols)}")
print(f"   • Clientes: {len(X_inference):,}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Realizar Predicciones

# COMMAND ----------

print("🎯 ETAPA 6: PREDICCIONES\n" + "="*60 + "\n")

# Predicción de clase (0 o 1)
predictions = model.predict(X_inference)

# Probabilidades (útil para ranking)
try:
    probabilities = model.predict_proba(X_inference)
    prob_premium = probabilities[:, 1]  # Probabilidad de ser premium
    has_proba = True
except:
    prob_premium = None
    has_proba = False
    print("⚠️  Modelo no soporta probabilidades\n")

print(f"✅ Predicciones completadas")
print(f"   • Total clientes: {len(predictions):,}")
print(f"   • Predichos como premium: {sum(predictions):,} ({sum(predictions)/len(predictions)*100:.1f}%)")
print(f"   • Predichos como no-premium: {len(predictions) - sum(predictions):,} ({(1-sum(predictions)/len(predictions))*100:.1f}%)\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Crear Tabla de Resultados

# COMMAND ----------

print("📊 ETAPA 7: TABLA DE RESULTADOS\n" + "="*60 + "\n")

# Crear DataFrame con resultados
results_df = pd.DataFrame({
    'customer_id': customer_ids,
    'is_premium_predicted': predictions,
    'premium_label': ['PREMIUM' if p == 1 else 'REGULAR' for p in predictions]
})

# Agregar probabilidades si están disponibles
if has_proba:
    results_df['premium_probability'] = prob_premium
    results_df['confidence'] = results_df['premium_probability'].apply(
        lambda x: 'HIGH' if x > 0.8 or x < 0.2 else 'MEDIUM' if x > 0.6 or x < 0.4 else 'LOW'
    )

# Agregar metadata
results_df['prediction_date'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
results_df['model_run_id'] = BEST_MODEL_RUN_ID
results_df['dataset_source'] = dataset_name

print(f"✅ Tabla de resultados creada: {results_df.shape}\n")

# Mostrar primeras filas
print("📋 Primeras 10 predicciones:")
print(results_df.head(10))
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Estadísticas de Predicción

# COMMAND ----------

print("📈 ETAPA 8: ESTADÍSTICAS\n" + "="*60 + "\n")

# Distribución de predicciones
print("🎯 Distribución de predicciones:")
print(results_df['premium_label'].value_counts())
print()

if has_proba:
    print("📊 Estadísticas de probabilidad:")
    print(results_df['premium_probability'].describe())
    print()
    
    print("🎚️  Distribución de confianza:")
    print(results_df['confidence'].value_counts())
    print()

# Top 10 clientes más probables de ser premium
if has_proba:
    print("🏆 Top 10 clientes con mayor probabilidad PREMIUM:")
    top_premium = results_df.nlargest(10, 'premium_probability')[
        ['customer_id', 'premium_label', 'premium_probability', 'confidence']
    ]
    print(top_premium)
    print()

# COMMAND ----------


##########################################################################
# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Guardar Resultados

# COMMAND ----------

print("💾 ETAPA 9: GUARDAR RESULTADOS\n" + "="*60 + "\n")

# Crear volume si no existe
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.predictions")
    print("✅ Volume 'predictions' verificado\n")
except:
    pass

# Generar nombre con timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_table = f"{RESULTS_PATH}predictions_{timestamp}/"

# ═══════════════════════════════════════════════════════════════
# OPCIÓN 1: Guardar solo como Delta Table (RECOMENDADO)
# ═══════════════════════════════════════════════════════════════

spark.createDataFrame(results_df).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_table)

print(f"✅ Delta Table guardada: {output_table}")

# ═══════════════════════════════════════════════════════════════
# OPCIÓN 2: Guardar CSV usando Spark (CORREGIDO)
# ═══════════════════════════════════════════════════════════════

output_csv_path = f"{RESULTS_PATH}predictions_{timestamp}_csv/"

# Convertir a Spark DataFrame y guardar como CSV
spark.createDataFrame(results_df).coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(output_csv_path)

print(f"✅ CSV guardado en: {output_csv_path}")

# Opcional: Renombrar el archivo CSV generado
try:
    csv_files = dbutils.fs.ls(output_csv_path)
    csv_file = [f for f in csv_files if f.name.endswith('.csv')][0]
    final_csv_path = f"{RESULTS_PATH}predictions_{timestamp}.csv"
    dbutils.fs.cp(csv_file.path, final_csv_path)
    print(f"✅ CSV renombrado: {final_csv_path}")
except Exception as e:
    print(f"⚠️  No se pudo renombrar CSV: {e}")

print()



# COMMAND ----------

# MAGIC %md
# MAGIC **💡 Cómo descargar el CSV:**
# MAGIC 
# MAGIC 1. **Desde Databricks:**
# MAGIC    ```python
# MAGIC    # Ver archivos disponibles
# MAGIC    display(dbutils.fs.ls(f"{RESULTS_PATH}"))
# MAGIC    ```
# MAGIC 
# MAGIC 2. **Descargar a local:**
# MAGIC    - Click derecho en el archivo → Download
# MAGIC 
# MAGIC 3. **O consultar directamente con Spark:**
# MAGIC    ```python
# MAGIC    predictions = spark.read.format("csv").option("header", "true").load(output_csv_path)
# MAGIC    display(predictions)
# MAGIC    ```
##########################################################################



# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Resumen Ejecutivo

# COMMAND ----------

duration = (datetime.now() - start_time).total_seconds()

print("\n" + "="*60)
print("✅ PREDICCIÓN COMPLETADA")
print("="*60)
print(f"\n📊 Resultados:")
print(f"   • Dataset: {dataset_name}")
print(f"   • Modelo: {BEST_MODEL_RUN_ID}")
print(f"   • Clientes analizados: {len(results_df):,}")
print(f"   • Clientes PREMIUM: {sum(predictions):,} ({sum(predictions)/len(predictions)*100:.1f}%)")
print(f"   • Clientes REGULAR: {len(predictions) - sum(predictions):,}")

if has_proba:
    print(f"\n📈 Confianza:")
    print(f"   • Alta: {(results_df['confidence'] == 'HIGH').sum():,}")
    print(f"   • Media: {(results_df['confidence'] == 'MEDIUM').sum():,}")
    print(f"   • Baja: {(results_df['confidence'] == 'LOW').sum():,}")

print(f"\n💾 Outputs:")
print(f"   • Delta: {output_table}")
print(f"   • CSV: {output_csv_path}")

print(f"\n⏱️  Duración: {duration:.2f} seg")
print("\n" + "="*60)

# COMMAND ----------

# ═══════════════════════════════════════════════════════════════
# AGREGAR AL FINAL DEL NOTEBOOK 11 (después de guardar predicciones)
# ═══════════════════════════════════════════════════════════════

# COMMAND ----------

# Exportar para Dashboard
DASHBOARD_PATH = "/Volumes/olist/olist_gold/dashboard/"

spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.dashboard")

# Tomar muestra de resultados para dashboard (primeros 1000)
inference_sample = results_df.head(1000)

inference_sample.to_csv(f"{DASHBOARD_PATH}inference_results_sample.csv", index=False)

print("✅ Datos exportados para dashboard")

# ═══════════════════════════════════════════════════════════════
# FIN - ESO ES TODO
# ═══════════════════════════════════════════════════════════════
# COMMAND ----------

